# Notebook 10: Variational Algorithms — VQE and QAOA

---

## Overview

Variational quantum algorithms are the leading candidates for achieving **quantum advantage on near-term (NISQ) devices**. Unlike Shor's or Grover's algorithms that require deep circuits and error correction, variational algorithms use **shallow parameterized circuits** optimized by a classical computer.

### What You Will Learn

1. The Variational Quantum Eigensolver (VQE) — finding ground state energies
2. The Rayleigh-Ritz variational principle
3. Ansatz design: hardware-efficient and chemically-inspired (UCCSD)
4. The Quantum Approximate Optimization Algorithm (QAOA)
5. MaxCut problem and QAOA formulation
6. The classical-quantum hybrid optimization loop
7. Full Qiskit implementations: VQE for H$_2$ and QAOA for MaxCut

### Why Variational Algorithms?

- **NISQ-friendly:** Short circuits $\to$ less noise accumulation
- **Flexible:** The classical optimizer can adapt to noise
- **Broad applicability:** Chemistry, optimization, machine learning, finance

## 1. The Variational Principle

### Rayleigh-Ritz Variational Principle

For any Hamiltonian $\hat{H}$ with ground state energy $E_0$, and **any** normalized trial state $|\psi(\vec{\theta})\rangle$:

$$\boxed{E_0 \leq \langle \psi(\vec{\theta}) | \hat{H} | \psi(\vec{\theta}) \rangle \equiv E(\vec{\theta})}$$

**Equality holds if and only if** $|\psi(\vec{\theta})\rangle$ is the ground state.

### Strategy

1. Parameterize a quantum state: $|\psi(\vec{\theta})\rangle = U(\vec{\theta})|0\rangle^{\otimes n}$
2. Measure the energy: $E(\vec{\theta}) = \langle 0|U^\dagger(\vec{\theta}) \hat{H} U(\vec{\theta})|0\rangle$
3. Minimize classically: $\vec{\theta}^* = \arg\min_{\vec{\theta}} E(\vec{\theta})$
4. $E(\vec{\theta}^*) \approx E_0$ — the ground state energy

### Proof of the Variational Principle

Let $\{|E_k\rangle\}$ be the eigenstates of $\hat{H}$ with energies $E_k$ (where $E_0 \leq E_1 \leq \cdots$).

Expand the trial state: $|\psi\rangle = \sum_k c_k |E_k\rangle$

$$\langle \psi|\hat{H}|\psi\rangle = \sum_k |c_k|^2 E_k \geq E_0 \sum_k |c_k|^2 = E_0$$

since $E_k \geq E_0$ for all $k$ and $\sum_k |c_k|^2 = 1$. $\blacksquare$

## 2. VQE — The Variational Quantum Eigensolver

### The Algorithm

```
┌─────────────────────────────────────────────┐
│  Classical Computer                         │
│  ┌─────────────────┐     ┌───────────────┐ │
│  │ Classical        │     │ Update θ      │ │
│  │ Optimizer        │────▶│ (COBYLA,      │ │
│  │ (minimize E(θ))  │     │  SPSA, etc.)  │ │
│  └────────▲─────────┘     └───────┬───────┘ │
│           │                       │         │
│     E(θ)  │                    θ  │         │
└───────────┼───────────────────────┼─────────┘
            │                       │
┌───────────┼───────────────────────┼─────────┐
│  Quantum  │   Computer            │         │
│           │                       ▼         │
│  ┌────────┴────────┐  ┌────────────────┐   │
│  │ Measure ⟨H⟩     │◀─│ Prepare        │   │
│  │ (Pauli terms)   │  │ |ψ(θ)⟩         │   │
│  └─────────────────┘  └────────────────┘   │
└─────────────────────────────────────────────┘
```

### Measuring the Hamiltonian

The Hamiltonian is decomposed into a sum of Pauli operators:

$$\hat{H} = \sum_i \alpha_i \hat{P}_i, \quad \hat{P}_i \in \{I, X, Y, Z\}^{\otimes n}$$

We measure each $\langle \hat{P}_i \rangle$ separately and combine:

$$E(\vec{\theta}) = \sum_i \alpha_i \langle \psi(\vec{\theta}) | \hat{P}_i | \psi(\vec{\theta}) \rangle$$

Each Pauli expectation value can be measured by:
1. Rotating the measurement basis (e.g., $H$ before measuring for $X$, $S^\dagger H$ for $Y$)
2. Measuring in the computational ($Z$) basis
3. Computing the expectation from shot statistics

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator
from qiskit.visualization import plot_histogram
from scipy.optimize import minimize
import matplotlib.pyplot as plt

print("Imports successful!")

## 3. Ansatz Design

The choice of **ansatz** (parameterized circuit) is crucial for VQE success.

### 3.1 Hardware-Efficient Ansatz

Uses gates native to the hardware:
- Layer of single-qubit rotations: $R_Y(\theta_i)$ or $R_Y(\theta_i)R_Z(\phi_i)$
- Layer of entangling gates: CNOT ladder or all-to-all CZ
- Repeat for $d$ layers

**Pros:** Easy to implement, low circuit depth
**Cons:** May have barren plateaus, no chemical intuition

### 3.2 UCCSD Ansatz (Unitary Coupled Cluster)

Inspired by classical computational chemistry:

$$|\psi(\vec{\theta})\rangle = e^{T(\vec{\theta}) - T^\dagger(\vec{\theta})} |\text{HF}\rangle$$

where $T = T_1 + T_2$ includes single and double excitations:

$$T_1 = \sum_{ia} \theta_i^a a_a^\dagger a_i, \quad T_2 = \sum_{ijab} \theta_{ij}^{ab} a_a^\dagger a_b^\dagger a_j a_i$$

**Pros:** Physically motivated, systematic improvement
**Cons:** Deep circuits, many parameters

In [ ]:
def hardware_efficient_ansatz(n_qubits, n_layers, params):
    """
    Build a hardware-efficient ansatz.
    
    Parameters:
    -----------
    n_qubits : int
    n_layers : int — number of repetition layers
    params : array — rotation angles
    
    Returns: QuantumCircuit
    """
    qc = QuantumCircuit(n_qubits)
    param_idx = 0
    
    for layer in range(n_layers):
        # Rotation layer: Ry and Rz on each qubit
        for q in range(n_qubits):
            qc.ry(params[param_idx], q)
            param_idx += 1
            qc.rz(params[param_idx], q)
            param_idx += 1
        
        # Entangling layer: CNOT ladder
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
    
    # Final rotation layer
    for q in range(n_qubits):
        qc.ry(params[param_idx], q)
        param_idx += 1
        qc.rz(params[param_idx], q)
        param_idx += 1
    
    return qc

def count_params_hea(n_qubits, n_layers):
    """Count parameters for hardware-efficient ansatz."""
    return 2 * n_qubits * (n_layers + 1)

# Example: 2 qubits, 2 layers
n_q, n_l = 2, 2
n_params = count_params_hea(n_q, n_l)
test_params = np.random.uniform(0, 2*np.pi, n_params)
ansatz_test = hardware_efficient_ansatz(n_q, n_l, test_params)

print(f"Hardware-Efficient Ansatz: {n_q} qubits, {n_l} layers, {n_params} parameters")
ansatz_test.draw('mpl', style='iqp', fold=False)
plt.title(f'Hardware-Efficient Ansatz ({n_q} qubits, {n_l} layers)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. VQE for the Hydrogen Molecule (H$_2$)

### The H$_2$ Hamiltonian

In the STO-3G basis with Jordan-Wigner mapping, the H$_2$ Hamiltonian (at a given bond length) can be written as:

$$\hat{H} = g_0 I + g_1 Z_0 + g_2 Z_1 + g_3 Z_0 Z_1 + g_4 X_0 X_1 + g_5 Y_0 Y_1$$

where the coefficients $g_i$ depend on the bond distance $d$.

At the equilibrium bond length $d \approx 0.735$ Angstrom:
- $g_0 = -1.0523732$
- $g_1 = 0.3979374$
- $g_2 = -0.3979374$
- $g_3 = -0.0112801$
- $g_4 = 0.1809312$
- $g_5 = 0.1809312$

The exact ground state energy is approximately $E_0 \approx -1.8572$ Hartree.

In [ ]:
# Define the H2 Hamiltonian at equilibrium bond length
def h2_hamiltonian(d=0.735):
    """
    Return the H2 Hamiltonian as a SparsePauliOp for a given bond length.
    Coefficients are pre-computed for STO-3G basis with Jordan-Wigner mapping.
    """
    # Approximate coefficients at d=0.735 Angstrom
    # These are from standard quantum chemistry calculations
    g0 = -1.0523732
    g1 = 0.3979374
    g2 = -0.3979374
    g3 = -0.0112801
    g4 = 0.1809312
    g5 = 0.1809312
    
    # Build Hamiltonian: H = g0*II + g1*IZ + g2*ZI + g3*ZZ + g4*XX + g5*YY
    hamiltonian = SparsePauliOp.from_list([
        ('II', g0),
        ('IZ', g1),
        ('ZI', g2),
        ('ZZ', g3),
        ('XX', g4),
        ('YY', g5)
    ])
    
    return hamiltonian

H_h2 = h2_hamiltonian()
print("H2 Hamiltonian:")
print(H_h2)
print(f"\nNumber of Pauli terms: {len(H_h2)}")

# Compute exact ground state energy
H_matrix = H_h2.to_matrix()
eigenvalues = np.linalg.eigvalsh(H_matrix)
E_exact = eigenvalues[0]
print(f"\nExact eigenvalues: {eigenvalues}")
print(f"Exact ground state energy: {E_exact:.6f} Hartree")

In [ ]:
def measure_expectation(circuit, hamiltonian, shots=8192):
    """
    Measure the expectation value ⟨ψ|H|ψ⟩ by decomposing H into Pauli terms.
    Uses the statevector simulator for exact results.
    """
    sv = Statevector.from_instruction(circuit)
    energy = sv.expectation_value(hamiltonian).real
    return energy

def vqe_cost_function(params, n_qubits, n_layers, hamiltonian):
    """
    The VQE cost function: build circuit with params, measure energy.
    """
    circuit = hardware_efficient_ansatz(n_qubits, n_layers, params)
    energy = measure_expectation(circuit, hamiltonian)
    return energy

print("VQE helper functions defined.")

In [ ]:
# Run VQE for H2
np.random.seed(42)

n_qubits = 2
n_layers = 2
n_params = count_params_hea(n_qubits, n_layers)

# Initial parameters
initial_params = np.random.uniform(-np.pi, np.pi, n_params)

# Track energy during optimization
energy_history = []

def callback(params):
    energy = vqe_cost_function(params, n_qubits, n_layers, H_h2)
    energy_history.append(energy)

# Record initial energy
callback(initial_params)

print(f"VQE for H2 Molecule")
print(f"==================")
print(f"Qubits: {n_qubits}")
print(f"Ansatz layers: {n_layers}")
print(f"Parameters: {n_params}")
print(f"Initial energy: {energy_history[0]:.6f} Hartree")
print(f"Exact energy:   {E_exact:.6f} Hartree")
print(f"\nOptimizing...")

# Run optimizer
result = minimize(
    vqe_cost_function,
    initial_params,
    args=(n_qubits, n_layers, H_h2),
    method='COBYLA',
    callback=callback,
    options={'maxiter': 500, 'rhobeg': 0.5}
)

print(f"\nOptimization complete!")
print(f"Final energy:   {result.fun:.6f} Hartree")
print(f"Exact energy:   {E_exact:.6f} Hartree")
print(f"Error:          {abs(result.fun - E_exact):.6f} Hartree")
print(f"Iterations:     {result.nfev}")

In [ ]:
# Plot convergence
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(energy_history, 'b-', linewidth=1.5, label='VQE energy')
ax.axhline(y=E_exact, color='red', linestyle='--', linewidth=2, label=f'Exact: {E_exact:.4f}')

ax.set_xlabel('Iteration', fontsize=13)
ax.set_ylabel('Energy (Hartree)', fontsize=13)
ax.set_title('VQE Convergence for H$_2$ Molecule', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Potential Energy Surface of H$_2$

By running VQE at different bond lengths, we can map out the potential energy surface (PES) — a key result in computational chemistry.

In [ ]:
def h2_hamiltonian_parametric(d):
    """
    Approximate H2 Hamiltonian coefficients as function of bond length d.
    These are simplified parameterizations for educational purposes.
    """
    # Simplified parametric model (approximate)
    g0 = -1.0523732 + 0.1 * (d - 0.735)**2
    g1 = 0.3979374 - 0.05 * (d - 0.735)
    g2 = -0.3979374 + 0.05 * (d - 0.735)
    g3 = -0.0112801 - 0.02 * (d - 0.735)**2
    g4 = 0.1809312 * np.exp(-0.5 * (d - 0.735)**2)
    g5 = 0.1809312 * np.exp(-0.5 * (d - 0.735)**2)
    
    hamiltonian = SparsePauliOp.from_list([
        ('II', g0),
        ('IZ', g1),
        ('ZI', g2),
        ('ZZ', g3),
        ('XX', g4),
        ('YY', g5)
    ])
    return hamiltonian

# Scan bond lengths
distances = np.linspace(0.3, 2.5, 25)
exact_energies = []
vqe_energies = []
hf_energies = []  # Hartree-Fock (just |01⟩)

print("Scanning H2 potential energy surface...")
print(f"{'d (Å)':<10} {'Exact':<15} {'VQE':<15} {'Error'}")
print("-" * 50)

best_params = np.random.uniform(-np.pi, np.pi, n_params)

for d in distances:
    H = h2_hamiltonian_parametric(d)
    H_mat = H.to_matrix()
    
    # Exact
    E_ex = np.linalg.eigvalsh(H_mat)[0]
    exact_energies.append(E_ex)
    
    # HF energy (|01⟩ state)
    hf_qc = QuantumCircuit(2)
    hf_qc.x(0)  # |01⟩
    E_hf = measure_expectation(hf_qc, H)
    hf_energies.append(E_hf)
    
    # VQE (warm-start from previous result)
    res = minimize(
        vqe_cost_function,
        best_params,
        args=(n_qubits, n_layers, H),
        method='COBYLA',
        options={'maxiter': 200}
    )
    best_params = res.x
    vqe_energies.append(res.fun)

print("Done!")

In [ ]:
# Plot potential energy surface
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(distances, exact_energies, 'k-', linewidth=2, label='Exact (FCI)')
ax.plot(distances, vqe_energies, 'bo', markersize=6, alpha=0.7, label='VQE')
ax.plot(distances, hf_energies, 'r--', linewidth=1.5, alpha=0.7, label='Hartree-Fock')

ax.set_xlabel('Bond Length d (Angstrom)', fontsize=13)
ax.set_ylabel('Energy (Hartree)', fontsize=13)
ax.set_title('H$_2$ Potential Energy Surface: VQE vs Exact', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Mark equilibrium
eq_idx = np.argmin(exact_energies)
ax.annotate(f'Equilibrium\nd={distances[eq_idx]:.2f} Å', 
            xy=(distances[eq_idx], exact_energies[eq_idx]),
            xytext=(distances[eq_idx]+0.5, exact_energies[eq_idx]+0.1),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=11)

plt.tight_layout()
plt.show()

## 6. QAOA — Quantum Approximate Optimization Algorithm

### The Problem: Combinatorial Optimization

Many important problems in computer science, logistics, and finance can be cast as:

$$\max_{z \in \{0,1\}^n} C(z)$$

where $C(z)$ is an **objective function** (also called the **cost function**).

### QAOA Approach (Farhi, Goldstone, Gutmann, 2014)

QAOA uses a parameterized quantum circuit to prepare a state that (approximately) maximizes $C(z)$.

### The QAOA Circuit

1. Start with $|+\rangle^{\otimes n} = H^{\otimes n}|0\rangle^{\otimes n}$
2. Alternate between two operators for $p$ rounds:
   - **Cost operator:** $U_C(\gamma) = e^{-i\gamma \hat{C}}$
   - **Mixer operator:** $U_M(\beta) = e^{-i\beta \hat{B}}$ where $\hat{B} = \sum_i X_i$

$$|\vec{\gamma}, \vec{\beta}\rangle = U_M(\beta_p) U_C(\gamma_p) \cdots U_M(\beta_1) U_C(\gamma_1) |+\rangle^{\otimes n}$$

3. Measure in computational basis
4. Classically optimize $\vec{\gamma}, \vec{\beta}$ to maximize $\langle \vec{\gamma}, \vec{\beta} | \hat{C} | \vec{\gamma}, \vec{\beta} \rangle$

### Theoretical Guarantees

- As $p \to \infty$, QAOA converges to the exact optimal solution
- For finite $p$, QAOA provides an **approximation** whose quality improves with $p$
- For MaxCut on 3-regular graphs, $p=1$ QAOA achieves approximation ratio $\geq 0.6924$

## 7. The MaxCut Problem

### Definition

Given a graph $G = (V, E)$, partition the vertices into two sets $S$ and $\bar{S}$ to **maximize** the number of edges between them.

### As an Optimization Problem

Assign $z_i \in \{0, 1\}$ to each vertex. The cost function is:

$$C(z) = \sum_{(i,j) \in E} z_i(1 - z_j) + z_j(1 - z_i) = \sum_{(i,j) \in E} z_i \oplus z_j$$

### Ising Formulation

Using spin variables $s_i = 1 - 2z_i \in \{+1, -1\}$:

$$C = \sum_{(i,j) \in E} \frac{1 - s_i s_j}{2} = \frac{|E|}{2} - \frac{1}{2}\sum_{(i,j) \in E} s_i s_j$$

As a quantum Hamiltonian (replacing $s_i \to Z_i$):

$$\hat{C} = \sum_{(i,j) \in E} \frac{1 - Z_i Z_j}{2}$$

Maximizing $C$ is equivalent to finding the ground state of $-\hat{C}$ (or equivalently, the highest energy state of $\hat{C}$).

In [ ]:
# Define a graph for MaxCut
# Simple 4-node graph:
#   0 --- 1
#   |     |
#   3 --- 2

edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
n_nodes = 4

# Visualize the graph
fig, ax = plt.subplots(figsize=(5, 5))

# Node positions (square)
positions = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (0, 0)}

for (i, j) in edges:
    xi, yi = positions[i]
    xj, yj = positions[j]
    ax.plot([xi, xj], [yi, yj], 'k-', linewidth=2)

for node, (x, y) in positions.items():
    ax.plot(x, y, 'o', markersize=30, color='steelblue')
    ax.text(x, y, str(node), ha='center', va='center', fontsize=14, color='white', fontweight='bold')

ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_aspect('equal')
ax.set_title('MaxCut Graph (4 nodes)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

# Brute force MaxCut
def maxcut_value(z, edges):
    """Compute the MaxCut value for a given bitstring."""
    return sum(z[i] != z[j] for i, j in edges)

print("\nBrute force MaxCut:")
print(f"{'Partition':<15} {'Cut value'}")
print("-" * 25)

max_cut = 0
max_partition = None
for i in range(2**n_nodes):
    z = [int(b) for b in format(i, f'0{n_nodes}b')]
    cut = maxcut_value(z, edges)
    print(f"{z}  →  {cut}")
    if cut > max_cut:
        max_cut = cut
        max_partition = z

print(f"\nOptimal cut: {max_cut} (partition: {max_partition})")

In [ ]:
def maxcut_hamiltonian(edges, n_nodes):
    """
    Build the MaxCut Hamiltonian: C = Σ (1 - Z_i Z_j) / 2
    """
    pauli_list = []
    for (i, j) in edges:
        # (1 - Z_i Z_j) / 2
        # Identity term
        pauli_list.append(('I' * n_nodes, 0.5))
        
        # -Z_i Z_j / 2
        zz_str = ['I'] * n_nodes
        zz_str[n_nodes - 1 - i] = 'Z'  # Qiskit uses reverse ordering
        zz_str[n_nodes - 1 - j] = 'Z'
        pauli_list.append((''.join(zz_str), -0.5))
    
    hamiltonian = SparsePauliOp.from_list(pauli_list).simplify()
    return hamiltonian

C_hat = maxcut_hamiltonian(edges, n_nodes)
print("MaxCut Hamiltonian:")
print(C_hat)

In [ ]:
def qaoa_circuit(edges, n_nodes, gamma, beta, p=1):
    """
    Build a QAOA circuit for MaxCut.
    
    Parameters:
    -----------
    edges : list of tuples
    n_nodes : int
    gamma : array of length p — cost layer parameters
    beta : array of length p — mixer layer parameters
    p : int — number of QAOA layers
    """
    qc = QuantumCircuit(n_nodes)
    
    # Initial state: |+⟩^n
    qc.h(range(n_nodes))
    
    for layer in range(p):
        qc.barrier()
        
        # Cost operator: U_C(γ) = exp(-iγC)
        # For MaxCut: exp(-iγ(1 - Z_i Z_j)/2) = exp(-iγ/2) exp(iγ Z_i Z_j / 2)
        # The ZZ interaction: exp(iγ Z_i Z_j / 2) = CNOT · Rz(γ) · CNOT
        for (i, j) in edges:
            qc.cx(i, j)
            qc.rz(gamma[layer], j)
            qc.cx(i, j)
        
        qc.barrier()
        
        # Mixer operator: U_M(β) = exp(-iβΣX_i) = Π exp(-iβ X_i) = Π Rx(2β)
        for i in range(n_nodes):
            qc.rx(2 * beta[layer], i)
    
    return qc

# Example QAOA circuit (p=1)
gamma_test = [0.5]
beta_test = [0.3]
qc_qaoa = qaoa_circuit(edges, n_nodes, gamma_test, beta_test, p=1)

qc_qaoa.draw('mpl', style='iqp', fold=False)
plt.title('QAOA Circuit for MaxCut (p=1)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def qaoa_expectation(params, edges, n_nodes, p, hamiltonian):
    """
    Compute the QAOA expectation value ⟨γ,β|C|γ,β⟩.
    """
    gamma = params[:p]
    beta = params[p:]
    
    qc = qaoa_circuit(edges, n_nodes, gamma, beta, p)
    sv = Statevector.from_instruction(qc)
    expectation = sv.expectation_value(hamiltonian).real
    
    return -expectation  # Negative because we minimize (but want to maximize cut)

print("QAOA expectation function defined.")

In [ ]:
# Run QAOA optimization for p=1
p = 1
n_qaoa_params = 2 * p  # gamma and beta

# Track cost
qaoa_history = []

def qaoa_callback(params):
    cost = qaoa_expectation(params, edges, n_nodes, p, C_hat)
    qaoa_history.append(-cost)  # Store actual MaxCut value

print(f"QAOA for MaxCut (p={p})")
print("=" * 40)

# Optimize
initial_qaoa_params = np.random.uniform(0, np.pi, n_qaoa_params)
qaoa_callback(initial_qaoa_params)

qaoa_result = minimize(
    qaoa_expectation,
    initial_qaoa_params,
    args=(edges, n_nodes, p, C_hat),
    method='COBYLA',
    callback=qaoa_callback,
    options={'maxiter': 300}
)

optimal_gamma = qaoa_result.x[:p]
optimal_beta = qaoa_result.x[p:]

print(f"Optimal gamma: {optimal_gamma}")
print(f"Optimal beta:  {optimal_beta}")
print(f"Optimal ⟨C⟩:   {-qaoa_result.fun:.4f}")
print(f"True max cut:  {max_cut}")
print(f"Approximation ratio: {-qaoa_result.fun / max_cut:.4f}")

In [ ]:
# Sample from the optimal QAOA state
optimal_qc = qaoa_circuit(edges, n_nodes, optimal_gamma, optimal_beta, p)
optimal_qc.measure_all()

sim = AerSimulator()
compiled = transpile(optimal_qc, sim)
result = sim.run(compiled, shots=4096).result()
counts = result.get_counts()

# Analyze results
print("QAOA Sampling Results:")
print(f"{'Bitstring':<12} {'Count':<8} {'Cut Value':<12} {'Optimal?'}")
print("-" * 40)

for bitstring in sorted(counts, key=counts.get, reverse=True):
    z = [int(b) for b in bitstring[::-1]]  # Reverse for Qiskit ordering
    cut = maxcut_value(z, edges)
    is_optimal = '*' if cut == max_cut else ''
    print(f"{bitstring:<12} {counts[bitstring]:<8} {cut:<12} {is_optimal}")

# Success probability
optimal_bitstrings = []
for i in range(2**n_nodes):
    z = [int(b) for b in format(i, f'0{n_nodes}b')]
    if maxcut_value(z, edges) == max_cut:
        optimal_bitstrings.append(format(i, f'0{n_nodes}b')[::-1])  # Reverse for Qiskit

success_count = sum(counts.get(bs, 0) for bs in optimal_bitstrings)
print(f"\nProbability of sampling optimal cut: {success_count / 4096:.4f}")

plot_histogram(counts)
plt.title('QAOA MaxCut: Sampling Distribution')
plt.show()

## 8. QAOA with Increasing Depth $p$

Let us see how the approximation ratio improves with the number of QAOA layers.

In [ ]:
# Compare QAOA performance for different p values
p_values = [1, 2, 3, 4, 5]
approx_ratios = []

print("QAOA Performance vs Depth p")
print("=" * 50)
print(f"{'p':<5} {'⟨C⟩':<10} {'Max Cut':<10} {'Approx Ratio':<15} {'Parameters'}")
print("-" * 50)

for p in p_values:
    n_params = 2 * p
    
    # Multiple random restarts
    best_val = float('inf')
    for _ in range(5):  # 5 random restarts
        x0 = np.random.uniform(0, np.pi, n_params)
        res = minimize(
            qaoa_expectation,
            x0,
            args=(edges, n_nodes, p, C_hat),
            method='COBYLA',
            options={'maxiter': 500}
        )
        if res.fun < best_val:
            best_val = res.fun
    
    exp_C = -best_val
    ratio = exp_C / max_cut
    approx_ratios.append(ratio)
    print(f"{p:<5} {exp_C:<10.4f} {max_cut:<10} {ratio:<15.4f} {n_params}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p_values, approx_ratios, 'bo-', linewidth=2, markersize=10)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Optimal')
ax.set_xlabel('QAOA depth $p$', fontsize=13)
ax.set_ylabel('Approximation ratio $\\langle C \\rangle / C_{max}$', fontsize=13)
ax.set_title('QAOA Approximation Ratio vs Circuit Depth', fontsize=14)
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(p_values)
plt.tight_layout()
plt.show()

## 9. QAOA Energy Landscape

For $p=1$, we can visualize the full energy landscape as a function of $(\gamma, \beta)$.

In [ ]:
# 2D energy landscape for QAOA p=1
gamma_range = np.linspace(0, 2*np.pi, 50)
beta_range = np.linspace(0, np.pi, 50)

energy_landscape = np.zeros((len(beta_range), len(gamma_range)))

for i, beta in enumerate(beta_range):
    for j, gamma in enumerate(gamma_range):
        params = np.array([gamma, beta])
        energy_landscape[i, j] = -qaoa_expectation(params, edges, n_nodes, 1, C_hat)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.contourf(gamma_range, beta_range, energy_landscape, levels=30, cmap='RdYlBu_r')
plt.colorbar(im, ax=ax, label='$\\langle C \\rangle$')

# Mark the optimal point
ax.plot(optimal_gamma[0], optimal_beta[0], 'k*', markersize=20, label='Optimum')

ax.set_xlabel('$\\gamma$', fontsize=14)
ax.set_ylabel('$\\beta$', fontsize=14)
ax.set_title('QAOA Energy Landscape ($p=1$, MaxCut)', fontsize=14)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

## 10. The Classical-Quantum Hybrid Loop

Both VQE and QAOA follow the same hybrid pattern:

### The Loop

1. **Quantum:** Prepare $|\psi(\vec{\theta})\rangle$ and measure $\langle \hat{H} \rangle$
2. **Classical:** Update $\vec{\theta}$ using an optimizer
3. **Repeat** until convergence

### Choice of Classical Optimizer

| Optimizer | Type | Pros | Cons |
|-----------|------|------|------|
| **COBYLA** | Gradient-free | No gradient circuits needed | Slow convergence |
| **Nelder-Mead** | Gradient-free | Simple, robust | Many function evaluations |
| **SPSA** | Stochastic gradient | Only 2 measurements per step | Noisy gradients |
| **L-BFGS-B** | Gradient-based | Fast convergence | Needs gradient circuits |
| **Adam** | Gradient-based | Adaptive learning rate | Needs many shots |
| **Parameter Shift** | Quantum gradient | Exact gradients on quantum hardware | 2 circuits per parameter |

### The Parameter Shift Rule

For a circuit with gate $R_P(\theta) = e^{-i\theta P/2}$ (where $P$ is a Pauli):

$$\frac{\partial \langle H \rangle}{\partial \theta} = \frac{\langle H \rangle_{\theta + \pi/2} - \langle H \rangle_{\theta - \pi/2}}{2}$$

This gives **exact gradients** using only two circuit evaluations per parameter!

In [ ]:
# Demonstrate the parameter shift rule
def parameter_shift_gradient(params, param_idx, n_qubits, n_layers, hamiltonian):
    """
    Compute the gradient of the VQE cost w.r.t. a single parameter
    using the parameter shift rule.
    """
    shift = np.pi / 2
    
    # Forward shift
    params_plus = params.copy()
    params_plus[param_idx] += shift
    E_plus = vqe_cost_function(params_plus, n_qubits, n_layers, hamiltonian)
    
    # Backward shift
    params_minus = params.copy()
    params_minus[param_idx] -= shift
    E_minus = vqe_cost_function(params_minus, n_qubits, n_layers, hamiltonian)
    
    return (E_plus - E_minus) / 2

# Compare with numerical gradient
test_params = np.random.uniform(-np.pi, np.pi, count_params_hea(2, 2))

print("Parameter Shift Rule vs Numerical Gradient")
print("=" * 55)
print(f"{'Param':<8} {'Shift Rule':<15} {'Numerical':<15} {'Match?'}")
print("-" * 55)

for i in range(min(6, len(test_params))):
    # Parameter shift gradient
    grad_ps = parameter_shift_gradient(test_params, i, 2, 2, H_h2)
    
    # Numerical gradient
    eps = 1e-5
    p_up = test_params.copy(); p_up[i] += eps
    p_dn = test_params.copy(); p_dn[i] -= eps
    grad_num = (vqe_cost_function(p_up, 2, 2, H_h2) - vqe_cost_function(p_dn, 2, 2, H_h2)) / (2 * eps)
    
    match = np.isclose(grad_ps, grad_num, atol=1e-4)
    print(f"θ_{i:<4}  {grad_ps:<15.6f} {grad_num:<15.6f} {'Yes' if match else 'No'}")

## 11. Barren Plateaus — A Challenge for Variational Algorithms

A major challenge for variational algorithms is the **barren plateau** phenomenon: as the number of qubits grows, the gradient of a random parameterized circuit **vanishes exponentially**.

$$\text{Var}\left[\frac{\partial \langle H \rangle}{\partial \theta_k}\right] \leq F(n) \to 0 \text{ exponentially in } n$$

### Mitigations

1. **Problem-inspired ansatze** (like UCCSD) instead of random hardware-efficient circuits
2. **Layer-wise training** — optimize one layer at a time
3. **Smart initialization** — start near a classically computed solution
4. **Local cost functions** — use local Pauli terms instead of global observables
5. **Shallow circuits** — keep depth $O(\log n)$

In [ ]:
# Demonstrate barren plateaus: gradient variance vs number of qubits
def measure_gradient_variance(n_qubits, n_layers, hamiltonian, n_samples=50):
    """Estimate the variance of the gradient for random parameters."""
    n_params = count_params_hea(n_qubits, n_layers)
    gradients = []
    
    for _ in range(n_samples):
        params = np.random.uniform(-np.pi, np.pi, n_params)
        grad = parameter_shift_gradient(params, 0, n_qubits, n_layers, hamiltonian)
        gradients.append(grad)
    
    return np.var(gradients)

# Test for different qubit counts
qubit_counts = [2, 3, 4, 5, 6]
grad_variances = []

print("Barren Plateau Analysis")
print("=" * 40)

for nq in qubit_counts:
    # Create a simple Hamiltonian: sum of all Z_i
    pauli_list = []
    for i in range(nq):
        z_str = ['I'] * nq
        z_str[nq - 1 - i] = 'Z'
        pauli_list.append((''.join(z_str), 1.0 / nq))
    H = SparsePauliOp.from_list(pauli_list)
    
    var = measure_gradient_variance(nq, 3, H, n_samples=100)
    grad_variances.append(var)
    print(f"n={nq}: Var[∂⟨H⟩/∂θ] = {var:.6f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(qubit_counts, grad_variances, 'ro-', linewidth=2, markersize=10)
ax.set_xlabel('Number of qubits', fontsize=13)
ax.set_ylabel('Var[$\\partial \\langle H \\rangle / \\partial \\theta$]', fontsize=13)
ax.set_title('Barren Plateau: Gradient Variance vs Qubits', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xticks(qubit_counts)
plt.tight_layout()
plt.show()

print("\nThe gradient variance decreases exponentially with qubit count —")
print("this is the barren plateau phenomenon!")

## 12. A Larger QAOA Example — 6-Node Graph

Let us apply QAOA to a larger MaxCut problem.

In [ ]:
# 6-node graph (random 3-regular graph)
edges_6 = [(0, 1), (0, 3), (0, 5), (1, 2), (1, 4), (2, 3), (2, 5), (3, 4), (4, 5)]
n_nodes_6 = 6

# Find optimal MaxCut by brute force
max_cut_6 = 0
optimal_partitions_6 = []

for i in range(2**n_nodes_6):
    z = [int(b) for b in format(i, f'0{n_nodes_6}b')]
    cut = maxcut_value(z, edges_6)
    if cut > max_cut_6:
        max_cut_6 = cut
        optimal_partitions_6 = [z]
    elif cut == max_cut_6:
        optimal_partitions_6.append(z)

print(f"6-node graph: {len(edges_6)} edges")
print(f"Optimal MaxCut: {max_cut_6}")
print(f"Number of optimal partitions: {len(optimal_partitions_6)}")

# Build Hamiltonian
C_hat_6 = maxcut_hamiltonian(edges_6, n_nodes_6)

# Run QAOA for different p
print(f"\nQAOA Results:")
print(f"{'p':<5} {'⟨C⟩':<10} {'Approx Ratio':<15}")
print("-" * 30)

for p in [1, 2, 3]:
    n_params = 2 * p
    best_val = float('inf')
    
    for _ in range(10):  # Random restarts
        x0 = np.random.uniform(0, np.pi, n_params)
        res = minimize(
            qaoa_expectation,
            x0,
            args=(edges_6, n_nodes_6, p, C_hat_6),
            method='COBYLA',
            options={'maxiter': 500}
        )
        if res.fun < best_val:
            best_val = res.fun
    
    exp_C = -best_val
    ratio = exp_C / max_cut_6
    print(f"{p:<5} {exp_C:<10.4f} {ratio:<15.4f}")

## 13. Comparison: VQE vs QAOA

| Feature | VQE | QAOA |
|---------|-----|------|
| **Goal** | Find ground state energy | Solve combinatorial optimization |
| **Ansatz** | Problem-agnostic (HEA) or chemistry-inspired (UCCSD) | Problem-specific (cost + mixer) |
| **Parameters** | $O(n \cdot d)$ for $d$ layers | $2p$ (independent of problem size!) |
| **Depth** | Variable (depends on ansatz) | $O(p \cdot |E|)$ per layer |
| **Guarantees** | Upper bound on ground energy | Approaches optimal as $p \to \infty$ |
| **Applications** | Quantum chemistry, materials | MaxCut, portfolio optimization, scheduling |

### Current Status (2025-2026)

- VQE has been demonstrated for molecules up to ~20 qubits on real hardware
- QAOA has shown advantages on specific structured problems
- Error mitigation techniques (ZNE, PEC, etc.) are crucial for near-term results
- The quest for **quantum advantage** with variational algorithms continues

## 14. Exercises

1. **VQE for LiH:** Extend the VQE implementation to the LiH molecule (4 qubits). Compare hardware-efficient ansatz with a chemically motivated ansatz.

2. **QAOA for weighted MaxCut:** Modify the QAOA implementation to handle weighted edges. Test on a graph where each edge has a different weight.

3. **Optimizer comparison:** Run VQE for H$_2$ using COBYLA, Nelder-Mead, and SPSA. Compare convergence speed and final accuracy.

4. **Noise study:** Add depolarizing noise to the VQE circuit and study how the ground state energy estimate degrades. Implement simple error mitigation (e.g., zero-noise extrapolation).

5. **QAOA for MaxSAT:** Formulate a simple 3-SAT problem as a QAOA instance and solve it.

## 15. Summary

| Algorithm | Key Idea | Application |
|-----------|----------|-------------|
| **VQE** | Variational principle: $E_0 \leq \langle\psi(\theta)|H|\psi(\theta)\rangle$ | Quantum chemistry, materials |
| **QAOA** | Alternating cost/mixer unitaries, depth $p$ | Combinatorial optimization |
| **Both** | Classical-quantum hybrid loop, NISQ-friendly | Near-term quantum advantage |

### Key Takeaways

1. Variational algorithms trade circuit depth for classical optimization overhead
2. Ansatz choice is critical — too expressive $\to$ barren plateaus, too restrictive $\to$ poor approximation
3. The parameter shift rule enables exact quantum gradients
4. These algorithms are the best candidates for useful quantum computing on current hardware